In [1]:
from IPython.display import clear_output
clear_output(wait=True)

import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

import ipywidgets as widgets
from ipywidgets import HBox, VBox, Output
from IPython.display import display
import plotly.express as px
import folium
from folium.plugins import HeatMap

PROJECT_ROOT = "/Users/sudhanshusahu/BS2"
OUTPUT_DIR   = os.path.join(PROJECT_ROOT, "outputs")
MODELS_DIR   = os.path.join(OUTPUT_DIR, "models")

df    = pd.read_csv(os.path.join(OUTPUT_DIR, "clustered_street.csv"))
df['Month'] = pd.to_datetime(df['Month'])
model = joblib.load(os.path.join(MODELS_DIR, "rf_crime_model.pkl"))
le    = joblib.load(os.path.join(MODELS_DIR, "label_encoder.pkl"))

In [2]:
display(widgets.HTML("""
<div style="font-family: Arial;">

  <div style="background: linear-gradient(135deg, #1a1a2e, #16213e, #0f3460);
              padding: 30px; border-radius: 12px; color: white; margin-bottom: 20px;">
    <h1 style="font-size: 28px; margin: 0 0 10px 0;">🚔 Norfolk Crime: Can We Predict It?</h1>
    <p style="font-size: 16px; color: #a0aec0; margin: 0 0 20px 0;">
        A data-driven investigation into 69,103 crimes across Norfolk 
        (March 2025 – March 2026)
    </p>
    <div style="background: rgba(255,255,255,0.1); padding: 15px; 
                border-radius: 8px; border-left: 4px solid #e53e3e;">
        <p style="font-size: 18px; color: white; margin: 0; font-style: italic;">
            "More than 6 in every 10 crimes reported in Norfolk 
            had no positive outcome in 2025–26."
        </p>
    </div>
  </div>

  <p style="font-size: 15px; color: #444; margin: 0 0 15px 0; font-weight: bold;">
      This dashboard answers 3 questions:
  </p>

  <div style="display:flex; gap:12px; flex-wrap:wrap; margin-bottom: 20px;">
    <div style="flex:1; min-width:200px; background:#fff5f5; 
                border-left: 4px solid #e53e3e;
                padding: 15px; border-radius: 8px;">
        <p style="font-size: 24px; margin: 0;">📍</p>
        <p style="font-size: 15px; font-weight: bold; color: #1a1a2e; margin: 8px 0 4px 0;">
            WHERE does crime happen?
        </p>
        <p style="font-size: 12px; color: #666; margin: 0;">
            Is crime spread evenly across Norfolk 
            or concentrated in specific hotspots?
        </p>
    </div>
    <div style="flex:1; min-width:200px; background:#fffff0;
                border-left: 4px solid #d69e2e;
                padding: 15px; border-radius: 8px;">
        <p style="font-size: 24px; margin: 0;">📅</p>
        <p style="font-size: 15px; font-weight: bold; color: #1a1a2e; margin: 8px 0 4px 0;">
            WHEN does crime happen?
        </p>
        <p style="font-size: 12px; color: #666; margin: 0;">
            Does crime follow seasonal or 
            monthly patterns we can anticipate?
        </p>
    </div>
    <div style="flex:1; min-width:200px; background:#f0fff4;
                border-left: 4px solid #38a169;
                padding: 15px; border-radius: 8px;">
        <p style="font-size: 24px; margin: 0;">🤖</p>
        <p style="font-size: 15px; font-weight: bold; color: #1a1a2e; margin: 8px 0 4px 0;">
            CAN WE PREDICT IT?
        </p>
        <p style="font-size: 12px; color: #666; margin: 0;">
            Can a machine learning model predict 
            crime type from location and time alone?
        </p>
    </div>
  </div>

  <p style="font-size: 12px; color: #999; margin: 0; text-align: center;">
      📂 Data source: data.police.uk — Norfolk Constabulary — 
      Street level crime — Mar 2025 to Mar 2026
  </p>

</div>
"""))

HTML(value='\n<div style="font-family: Arial;">\n\n  <div style="background: linear-gradient(135deg, #1a1a2e, …

In [3]:
display(widgets.HTML("""
<div style="font-family: Arial; margin: 20px 0;">

  <p style="font-size: 18px; font-weight: bold; color: #1a1a2e; margin: 0 0 5px 0;">
      📊 First, let's understand the scale of the problem
  </p>
  <p style="font-size: 13px; color: #666; margin: 0 0 20px 0;">
      These are not just numbers — each one represents a real incident 
      reported to Norfolk Constabulary.
  </p>

  <div style="display:flex; gap:12px; flex-wrap:wrap; margin-bottom:12px;">

    <div style="flex:1; min-width:180px; background:white;
                border-left: 5px solid #3182ce;
                padding: 18px; border-radius: 8px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.08);">
      <p style="color:#666; font-size:12px; margin:0 0 6px 0;">Total crimes recorded</p>
      <p style="color:#1a1a2e; font-size:30px; font-weight:bold; margin:0;">69,103</p>
      <p style="color:#3182ce; font-size:11px; margin:6px 0 0 0;">
          Mar 2025 – Mar 2026
      </p>
    </div>

    <div style="flex:1; min-width:180px; background:white;
                border-left: 5px solid #e53e3e;
                padding: 18px; border-radius: 8px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.08);">
      <p style="color:#666; font-size:12px; margin:0 0 6px 0;">Crimes with NO outcome</p>
      <p style="color:#e53e3e; font-size:30px; font-weight:bold; margin:0;">60.48%</p>
      <p style="color:#e53e3e; font-size:11px; margin:6px 0 0 0;">
          ⚠️ More than 6 in 10 go unresolved
      </p>
    </div>

    <div style="flex:1; min-width:180px; background:white;
                border-left: 5px solid #805ad5;
                padding: 18px; border-radius: 8px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.08);">
      <p style="color:#666; font-size:12px; margin:0 0 6px 0;">Unique crime locations</p>
      <p style="color:#1a1a2e; font-size:30px; font-weight:bold; margin:0;">7,446</p>
      <p style="color:#805ad5; font-size:11px; margin:6px 0 0 0;">
          Spread across Norfolk
      </p>
    </div>

    <div style="flex:1; min-width:180px; background:white;
                border-left: 5px solid #dd6b20;
                padding: 18px; border-radius: 8px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.08);">
      <p style="color:#666; font-size:12px; margin:0 0 6px 0;">Hotspot clusters found</p>
      <p style="color:#1a1a2e; font-size:30px; font-weight:bold; margin:0;">127</p>
      <p style="color:#dd6b20; font-size:11px; margin:6px 0 0 0;">
          📍 Crime is NOT random
      </p>
    </div>

  </div>

  <div style="background: #fff5f5; border: 1px solid #fed7d7;
              padding: 15px 20px; border-radius: 8px; margin-top: 10px;">
    <p style="font-size: 13px; color: #c53030; margin: 0;">
        💡 <b>What this tells us:</b> With 60.48% of crimes going unresolved 
        and crime concentrated in just 127 clusters, 
        smarter — not just harder — policing is needed. 
        Predicting WHERE and WHEN crime happens 
        could change this.
    </p>
  </div>

</div>
"""))

HTML(value='\n<div style="font-family: Arial; margin: 20px 0;">\n\n  <p style="font-size: 18px; font-weight: b…

In [4]:
display(widgets.HTML("""
<div style="font-family: Arial; margin: 20px 0;">

  <h2 style="font-size: 22px; color: #1a1a2e; margin: 0 0 5px 0;">
      📍 Question 1: WHERE does crime happen in Norfolk?
  </h2>
  <p style="font-size: 13px; color: #666; margin: 0 0 20px 0;">
      If crime were random, it would be spread evenly across Norfolk's 
      5,372 km². It is not.
  </p>

  <div style="background: #1a1a2e; color: white; padding: 20px; 
              border-radius: 10px; margin-bottom: 15px;">
    <p style="font-size: 14px; color: #a0aec0; margin: 0 0 15px 0; font-weight:bold;">
        🔍 KEY FINDING: Crime concentrates in just 3 cities
    </p>
    <div style="display:flex; gap:15px; flex-wrap:wrap;">
      <div style="flex:1; min-width:150px; text-align:center; 
                  background:rgba(229,62,62,0.2); padding:15px; border-radius:8px;
                  border: 1px solid #e53e3e;">
        <p style="font-size:13px; color:#fc8181; margin:0;">🔴 Norwich</p>
        <p style="font-size:32px; font-weight:bold; color:white; margin:5px 0;">23,941</p>
        <p style="font-size:12px; color:#a0aec0; margin:0;">crimes — <b style="color:#fc8181;">34.6%</b> of total</p>
      </div>
      <div style="flex:1; min-width:150px; text-align:center;
                  background:rgba(221,107,32,0.2); padding:15px; border-radius:8px;
                  border: 1px solid #dd6b20;">
        <p style="font-size:13px; color:#f6ad55; margin:0;">🟠 Great Yarmouth</p>
        <p style="font-size:32px; font-weight:bold; color:white; margin:5px 0;">10,060</p>
        <p style="font-size:12px; color:#a0aec0; margin:0;">crimes — <b style="color:#f6ad55;">14.6%</b> of total</p>
      </div>
      <div style="flex:1; min-width:150px; text-align:center;
                  background:rgba(214,158,46,0.2); padding:15px; border-radius:8px;
                  border: 1px solid #d69e2e;">
        <p style="font-size:13px; color:#f6e05e; margin:0;">🟡 King's Lynn</p>
        <p style="font-size:32px; font-weight:bold; color:white; margin:5px 0;">5,832</p>
        <p style="font-size:12px; color:#a0aec0; margin:0;">crimes — <b style="color:#f6e05e;">8.4%</b> of total</p>
      </div>
      <div style="flex:1; min-width:150px; text-align:center;
                  background:rgba(56,161,105,0.2); padding:15px; border-radius:8px;
                  border: 1px solid #38a169;">
        <p style="font-size:13px; color:#68d391; margin:0;">🟢 Rest of Norfolk</p>
        <p style="font-size:32px; font-weight:bold; color:white; margin:5px 0;">29,270</p>
        <p style="font-size:12px; color:#a0aec0; margin:0;">crimes across <b style="color:#68d391;">124</b> smaller clusters</p>
      </div>
    </div>
    <div style="margin-top:15px; padding:12px; background:rgba(255,255,255,0.05); 
                border-radius:6px; border-left: 3px solid #68d391;">
      <p style="font-size:13px; color:#68d391; margin:0;">
          ✅ Just 3 cities account for <b>57.6% of all crime</b> in Norfolk. 
          This confirms Weisburd's Law — crime concentrates 
          in a small number of places.
      </p>
    </div>
  </div>

  <div style="background:#f7fafc; border:1px solid #e2e8f0;
              padding:15px 20px; border-radius:8px; margin: 15px 0;">
    <p style="font-size:13px; font-weight:bold; color:#1a1a2e; margin:0 0 8px 0;">
        🔬 How did we find these hotspots?
    </p>
    <p style="font-size:13px; color:#444; line-height:1.8; margin:0;">
        We used an algorithm called <b>DBSCAN</b> 
        (Density-Based Spatial Clustering) — 
        think of it like a smart GPS that automatically 
        groups crimes that happen close together into clusters. 
        It found <b>127 crime hotspots</b> across Norfolk 
        without us telling it where to look. 
        It just followed the data. 
        <b>89.5% of all crimes</b> fell inside one of these clusters — 
        proving crime is not random but geographically concentrated.
    </p>
  </div>

  <p style="font-size: 14px; font-weight:bold; color:#1a1a2e; margin: 15px 0 8px 0;">
      🏢 And within those cities — crime concentrates in specific places:
  </p>
  <div style="display:flex; gap:10px; flex-wrap:wrap;">
    <div style="flex:1; min-width:140px; background:#fff5f5; padding:12px; 
                border-radius:8px; text-align:center;">
      <p style="font-size:20px; margin:0;">🅿️</p>
      <p style="font-size:13px; font-weight:bold; color:#1a1a2e; margin:4px 0;">Parking Areas</p>
      <p style="font-size:20px; font-weight:bold; color:#e53e3e; margin:0;">3,144</p>
      <p style="font-size:11px; color:#666; margin:4px 0 0 0;">Most dangerous spot</p>
    </div>
    <div style="flex:1; min-width:140px; background:#fff8f0; padding:12px;
                border-radius:8px; text-align:center;">
      <p style="font-size:20px; margin:0;">🛒</p>
      <p style="font-size:13px; font-weight:bold; color:#1a1a2e; margin:4px 0;">Supermarkets</p>
      <p style="font-size:20px; font-weight:bold; color:#dd6b20; margin:0;">2,871</p>
      <p style="font-size:11px; color:#666; margin:4px 0 0 0;">Shoplifting hotspot</p>
    </div>
    <div style="flex:1; min-width:140px; background:#fffff0; padding:12px;
                border-radius:8px; text-align:center;">
      <p style="font-size:20px; margin:0;">🏬</p>
      <p style="font-size:13px; font-weight:bold; color:#1a1a2e; margin:4px 0;">Shopping Areas</p>
      <p style="font-size:20px; font-weight:bold; color:#d69e2e; margin:0;">1,412</p>
      <p style="font-size:11px; color:#666; margin:4px 0 0 0;">Retail crime zone</p>
    </div>
    <div style="flex:1; min-width:140px; background:#f0fff4; padding:12px;
                border-radius:8px; text-align:center;">
      <p style="font-size:20px; margin:0;">⛽</p>
      <p style="font-size:13px; font-weight:bold; color:#1a1a2e; margin:4px 0;">Petrol Stations</p>
      <p style="font-size:20px; font-weight:bold; color:#38a169; margin:0;">1,309</p>
      <p style="font-size:11px; color:#666; margin:4px 0 0 0;">Vehicle crime hub</p>
    </div>
    <div style="flex:1; min-width:140px; background:#ebf8ff; padding:12px;
                border-radius:8px; text-align:center;">
      <p style="font-size:20px; margin:0;">🍸</p>
      <p style="font-size:13px; font-weight:bold; color:#1a1a2e; margin:4px 0;">Nightclubs</p>
      <p style="font-size:20px; font-weight:bold; color:#3182ce; margin:0;">829</p>
      <p style="font-size:11px; color:#666; margin:4px 0 0 0;">Violence hotspot</p>
    </div>
  </div>

</div>
"""))

HTML(value='\n<div style="font-family: Arial; margin: 20px 0;">\n\n  <h2 style="font-size: 22px; color: #1a1a2…

In [5]:
import folium
from folium.plugins import HeatMap

section_map = widgets.HTML("""
<div style="font-family: Arial; font-size: 18px; font-weight: bold;
            color: #1a1a2e; padding: 15px 0 5px 0;
            border-bottom: 3px solid #805ad5; margin-bottom: 10px;">
    🗺️ Interactive Crime Hotspot Map
</div>
""")

df_sample   = df.sample(5000, random_state=42)
heat_points = df_sample[['Latitude', 'Longitude']].values.tolist()

m = folium.Map(location=[52.63, 1.10], zoom_start=11, tiles='CartoDB positron')
HeatMap(heat_points, radius=12, blur=15, max_zoom=13).add_to(m)

map_note = widgets.HTML("""
<p style="font-family: Arial; font-size: 12px; color: #666; margin: 8px 0;">
    💡 Scroll to zoom in/out. Brighter = higher crime density. 
    5,000 sampled points. Norwich glows brightest — 34.6% of all crime.
</p>
""")

map_html = widgets.HTML(
    f'<div style="width:100%; height:500px; '
    f'border-radius:8px; overflow:hidden;">'
    f'{m._repr_html_()}</div>'
)

display(section_map, map_note, map_html)

HTML(value='\n<div style="font-family: Arial; font-size: 18px; font-weight: bold;\n            color: #1a1a2e;…

HTML(value='\n<p style="font-family: Arial; font-size: 12px; color: #666; margin: 8px 0;">\n    💡 Scroll to zo…

HTML(value='<div style="width:100%; height:500px; border-radius:8px; overflow:hidden;"><div style="width:100%;…

In [6]:
section_when = widgets.HTML("""
<div style="font-family: Arial; margin: 20px 0;">
  <h2 style="font-size: 22px; color: #1a1a2e; margin: 0 0 5px 0;">
      📅 Question 2: WHEN does crime happen in Norfolk?
  </h2>
  <p style="font-size: 13px; color: #666; margin: 0 0 15px 0;">
      If we know crime peaks at certain times of year, 
      police can prepare — not just react.
  </p>
</div>
""")

out_when = Output()
with out_when:
    monthly = df.groupby('Month').size().reset_index(name='Count')

    fig = px.line(monthly, x='Month', y='Count',
                  markers=True,
                  color_discrete_sequence=['#3182ce'])

    fig.add_hline(y=monthly['Count'].mean(),
                  line_dash="dash", line_color="red",
                  annotation_text=f"Annual average: {int(monthly['Count'].mean()):,}",
                  annotation_position="top right")

    fig.update_layout(
        height=350,
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis_title='Month',
        yaxis_title='Number of Crimes',
        font=dict(family='Arial')
    )
    fig.show()

insight_when = widgets.HTML("""
<div style="display:flex; gap:12px; flex-wrap:wrap; margin-top:10px;">

  <div style="flex:1; min-width:180px; background:#fff5f5;
              border-left:4px solid #e53e3e;
              padding:12px; border-radius:8px;">
    <p style="font-size:12px; color:#c53030; font-weight:bold; margin:0;">
        ☀️ Summer Peak
    </p>
    <p style="font-size:22px; font-weight:bold; color:#1a1a2e; margin:4px 0;">
        July 2025
    </p>
    <p style="font-size:12px; color:#666; margin:0;">
        Busiest month — 6,049 crimes.
        More people outside = more opportunity.
    </p>
  </div>

  <div style="flex:1; min-width:180px; background:#ebf8ff;
              border-left:4px solid #3182ce;
              padding:12px; border-radius:8px;">
    <p style="font-size:12px; color:#2b6cb0; font-weight:bold; margin:0;">
        ❄️ Winter Trough
    </p>
    <p style="font-size:22px; font-weight:bold; color:#1a1a2e; margin:4px 0;">
        Feb 2026
    </p>
    <p style="font-size:12px; color:#666; margin:0;">
        Quietest month — 4,764 crimes.
        21% fewer than peak summer.
    </p>
  </div>

  <div style="flex:1; min-width:180px; background:#f0fff4;
              border-left:4px solid #38a169;
              padding:12px; border-radius:8px;">
    <p style="font-size:12px; color:#276749; font-weight:bold; margin:0;">
        📈 Spring Surge
    </p>
    <p style="font-size:22px; font-weight:bold; color:#1a1a2e; margin:4px 0;">
        Mar 2026
    </p>
    <p style="font-size:12px; color:#666; margin:0;">
        Crime rising again — 5,680.
        Seasonal cycle repeating.
    </p>
  </div>

  <div style="flex:1; min-width:180px; background:#fffff0;
              border-left:4px solid #d69e2e;
              padding:12px; border-radius:8px;">
    <p style="font-size:12px; color:#975a16; font-weight:bold; margin:0;">
        💡 Key Insight
    </p>
    <p style="font-size:15px; font-weight:bold; color:#1a1a2e; margin:4px 0;">
        Crime is seasonal
    </p>
    <p style="font-size:12px; color:#666; margin:0;">
        Police CAN anticipate busy periods
        and deploy resources proactively.
    </p>
  </div>

</div>
""")

display(section_when, out_when, insight_when)

HTML(value='\n<div style="font-family: Arial; margin: 20px 0;">\n  <h2 style="font-size: 22px; color: #1a1a2e;…

Output()

HTML(value='\n<div style="display:flex; gap:12px; flex-wrap:wrap; margin-top:10px;">\n\n  <div style="flex:1; …

In [7]:
section_what = widgets.HTML("""
<div style="font-family: Arial; margin: 20px 0;">
  <h2 style="font-size: 22px; color: #1a1a2e; margin: 0 0 5px 0;">
      🔍 Question 3: WHAT types of crime are happening?
  </h2>
  <p style="font-size: 13px; color: #666; margin: 0 0 15px 0;">
      Not all crime is equal. Understanding WHAT is happening 
      helps police decide WHERE to focus their limited resources.
  </p>
</div>
""")

out_what = Output()
with out_what:
    crime_counts = df['Crime type'].value_counts().reset_index()
    crime_counts.columns = ['Crime Type', 'Count']

    fig = px.bar(crime_counts,
                 x='Count', y='Crime Type',
                 orientation='h',
                 color='Count',
                 color_continuous_scale='RdYlGn_r')

    fig.update_layout(
        height=420,
        showlegend=False,
        plot_bgcolor='white',
        paper_bgcolor='white',
        coloraxis_showscale=False,
        font=dict(family='Arial'),
        xaxis_title='Number of Incidents',
        yaxis_title=''
    )
    fig.show()

insight_what = widgets.HTML("""
<div style="display:flex; gap:12px; flex-wrap:wrap; margin-top:10px;">

  <div style="flex:1; min-width:180px; background:#fff5f5;
              border-left:4px solid #e53e3e;
              padding:12px; border-radius:8px;">
    <p style="font-size:12px; color:#c53030; font-weight:bold; margin:0;">
        ⚠️ Biggest Problem
    </p>
    <p style="font-size:16px; font-weight:bold; color:#1a1a2e; margin:4px 0;">
        Violence & Sexual Offences
    </p>
    <p style="font-size:12px; color:#666; margin:0;">
        29,850 incidents — 43% of ALL crime.
        The single biggest challenge for Norfolk Police.
    </p>
  </div>

  <div style="flex:1; min-width:180px; background:#fff8f0;
              border-left:4px solid #dd6b20;
              padding:12px; border-radius:8px;">
    <p style="font-size:12px; color:#c05621; font-weight:bold; margin:0;">
        🛒 Retail Crime Surge
    </p>
    <p style="font-size:16px; font-weight:bold; color:#1a1a2e; margin:4px 0;">
        Shoplifting — 6,072
    </p>
    <p style="font-size:12px; color:#666; margin:0;">
        Concentrated in supermarkets.
        Highly predictable by location.
    </p>
  </div>

  <div style="flex:1; min-width:180px; background:#f0fff4;
              border-left:4px solid #38a169;
              padding:12px; border-radius:8px;">
    <p style="font-size:12px; color:#276749; font-weight:bold; margin:0;">
        🏘️ Community Impact
    </p>
    <p style="font-size:16px; font-weight:bold; color:#1a1a2e; margin:4px 0;">
        Anti-social Behaviour — 7,460
    </p>
    <p style="font-size:12px; color:#666; margin:0;">
        Peaks in summer outdoors.
        Drops sharply in winter.
    </p>
  </div>

  <div style="flex:1; min-width:180px; background:#ebf8ff;
              border-left:4px solid #3182ce;
              padding:12px; border-radius:8px;">
    <p style="font-size:12px; color:#2b6cb0; font-weight:bold; margin:0;">
        💡 Key Insight
    </p>
    <p style="font-size:16px; font-weight:bold; color:#1a1a2e; margin:4px 0;">
        Crime type = location
    </p>
    <p style="font-size:12px; color:#666; margin:0;">
        Shoplifting happens at supermarkets.
        Violence happens near nightclubs.
        Location predicts crime type.
    </p>
  </div>

</div>
""")

display(section_what, out_what, insight_what)

HTML(value='\n<div style="font-family: Arial; margin: 20px 0;">\n  <h2 style="font-size: 22px; color: #1a1a2e;…

Output()

HTML(value='\n<div style="display:flex; gap:12px; flex-wrap:wrap; margin-top:10px;">\n\n  <div style="flex:1; …

In [8]:
section_predict = widgets.HTML("""
<div style="font-family: Arial; margin: 20px 0;">
  <h2 style="font-size: 22px; color: #1a1a2e; margin: 0 0 5px 0;">
      🤖 Question 4: Can we actually PREDICT crime?
  </h2>
  <p style="font-size: 13px; color: #666; margin: 0 0 15px 0;">
      We've seen WHERE, WHEN and WHAT. Now the big question — 
      can a machine learn these patterns and predict 
      what crime will happen at a given place and time?
  </p>
</div>
""")

out_predict = Output()
with out_predict:
    feature_names = ['Latitude', 'Longitude', 'Month', 'Season', 'Cluster']
    importances   = [0.4013, 0.3863, 0.1012, 0.0380, 0.0732]

    fig = px.bar(
        x=importances, y=feature_names,
        orientation='h',
        color=importances,
        color_continuous_scale='Blues',
        title='What does the model use to predict crime type?'
    )
    fig.update_layout(
        height=300,
        plot_bgcolor='white',
        paper_bgcolor='white',
        coloraxis_showscale=False,
        font=dict(family='Arial'),
        xaxis_title='Importance Score',
        yaxis_title=''
    )
    fig.show()

insight_predict = widgets.HTML("""
<div style="font-family: Arial; margin-top: 15px;">

  <div style="background:#f7fafc; border:1px solid #e2e8f0;
              padding:15px 20px; border-radius:8px; margin-bottom:15px;">
    <p style="font-size:13px; font-weight:bold; color:#1a1a2e; margin:0 0 8px 0;">
        🧠 How does the model work?
    </p>
    <p style="font-size:13px; color:#444; line-height:1.8; margin:0;">
        We trained a <b>Random Forest</b> — a machine learning model 
        that learns patterns from 55,000+ past crimes. 
        It looks at <b>where</b> a crime happened (latitude & longitude), 
        <b>when</b> it happened (month & season), 
        and <b>which cluster</b> the location belongs to — 
        then predicts what TYPE of crime it is.
    </p>
  </div>

  <div style="display:flex; gap:12px; flex-wrap:wrap; margin-bottom:15px;">

    <div style="flex:1; min-width:180px; background:#f0fff4;
                border-left:4px solid #38a169;
                padding:12px; border-radius:8px;">
      <p style="font-size:12px; color:#276749; font-weight:bold; margin:0;">
          ✅ Model Result
      </p>
      <p style="font-size:22px; font-weight:bold; color:#1a1a2e; margin:4px 0;">
          +20.46%
      </p>
      <p style="font-size:12px; color:#666; margin:0;">
          Better than random guessing.
          Macro F1 score: 0.3459
      </p>
    </div>

    <div style="flex:1; min-width:180px; background:#ebf8ff;
                border-left:4px solid #3182ce;
                padding:12px; border-radius:8px;">
      <p style="font-size:12px; color:#2b6cb0; font-weight:bold; margin:0;">
          📍 Location Power
      </p>
      <p style="font-size:22px; font-weight:bold; color:#1a1a2e; margin:4px 0;">
          79%
      </p>
      <p style="font-size:12px; color:#666; margin:0;">
          Of prediction comes from 
          latitude & longitude alone.
          WHERE = WHAT.
      </p>
    </div>

    <div style="flex:1; min-width:180px; background:#fffff0;
                border-left:4px solid #d69e2e;
                padding:12px; border-radius:8px;">
      <p style="font-size:12px; color:#975a16; font-weight:bold; margin:0;">
          🛒 Best Predicted
      </p>
      <p style="font-size:22px; font-weight:bold; color:#1a1a2e; margin:4px 0;">
          Shoplifting
      </p>
      <p style="font-size:12px; color:#666; margin:0;">
          F1 score: 0.67 — model 
          knows exactly where 
          shoplifting happens.
      </p>
    </div>

    <div style="flex:1; min-width:180px; background:#fff5f5;
                border-left:4px solid #e53e3e;
                padding:12px; border-radius:8px;">
      <p style="font-size:12px; color:#c53030; font-weight:bold; margin:0;">
          ❓ Hardest to Predict
      </p>
      <p style="font-size:22px; font-weight:bold; color:#1a1a2e; margin:4px 0;">
          Public Order
      </p>
      <p style="font-size:12px; color:#666; margin:0;">
          F1 score: 0.15 — happens 
          anywhere, anytime. 
          Hard to anticipate.
      </p>
    </div>

  </div>

  <div style="background: linear-gradient(135deg, #1a1a2e, #16213e);
              padding: 20px; border-radius: 10px; color: white;">
    <p style="font-size:14px; font-weight:bold; color:#68d391; margin:0 0 8px 0;">
        ✅ Answer to our Research Question:
    </p>
    <p style="font-size:14px; color:white; line-height:1.8; margin:0 0 10px 0;">
        <b>Yes — crime in Norfolk IS predictable.</b> 
        It clusters in specific places (Norwich, Great Yarmouth, King's Lynn), 
        peaks in summer, and follows location-based patterns 
        that a machine learning model can learn. 
        Location alone accounts for 79% of what type of crime occurs.
    </p>
    <p style="font-size:13px; color:#a0aec0; margin:0;">
        This means Norfolk Constabulary can use data — not just instinct — 
        to decide where to patrol, when to increase resources, 
        and which crime types to prioritise in which areas.
    </p>
  </div>

</div>
""")

display(section_predict, out_predict, insight_predict)

HTML(value='\n<div style="font-family: Arial; margin: 20px 0;">\n  <h2 style="font-size: 22px; color: #1a1a2e;…

Output()

HTML(value='\n<div style="font-family: Arial; margin-top: 15px;">\n\n  <div style="background:#f7fafc; border:…

In [9]:
display(widgets.HTML("""
<div style="font-family: Arial; margin: 20px 0;">

  <h2 style="font-size: 22px; color: #1a1a2e; margin: 0 0 5px 0;">
      🎯 So What? What Should Norfolk Police Actually Do?
  </h2>
  <p style="font-size: 13px; color: #666; margin: 0 0 20px 0;">
      Data without action is just noise. 
      Here is what our analysis recommends for Norfolk Constabulary.
  </p>

  <div style="display:flex; gap:12px; flex-wrap:wrap; margin-bottom:20px;">

    <div style="flex:1; min-width:220px; background:white;
                border-top: 4px solid #e53e3e;
                padding:15px; border-radius:8px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.06);">
      <p style="font-size:24px; margin:0;">1️⃣</p>
      <p style="font-size:14px; font-weight:bold; color:#1a1a2e; margin:8px 0 4px 0;">
          Focus patrols on the Big 3
      </p>
      <p style="font-size:12px; color:#666; line-height:1.7; margin:0;">
          Norwich, Great Yarmouth and King's Lynn 
          account for <b>57.6% of all crime</b>. 
          Targeted patrols in these cities — 
          especially parking areas and supermarkets — 
          would have the highest impact.
      </p>
    </div>

    <div style="flex:1; min-width:220px; background:white;
                border-top: 4px solid #dd6b20;
                padding:15px; border-radius:8px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.06);">
      <p style="font-size:24px; margin:0;">2️⃣</p>
      <p style="font-size:14px; font-weight:bold; color:#1a1a2e; margin:8px 0 4px 0;">
          Scale up in Summer
      </p>
      <p style="font-size:12px; color:#666; line-height:1.7; margin:0;">
          Crime peaks in July — 21% higher 
          than the winter low. 
          Norfolk Police should increase visible 
          patrols from May to September, 
          particularly in coastal areas 
          like Great Yarmouth.
      </p>
    </div>

    <div style="flex:1; min-width:220px; background:white;
                border-top: 4px solid #38a169;
                padding:15px; border-radius:8px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.06);">
      <p style="font-size:24px; margin:0;">3️⃣</p>
      <p style="font-size:14px; font-weight:bold; color:#1a1a2e; margin:8px 0 4px 0;">
          Partner with Retailers
      </p>
      <p style="font-size:12px; color:#666; line-height:1.7; margin:0;">
          Shoplifting at supermarkets is 
          highly predictable by location. 
          A dedicated retail crime partnership 
          between Norfolk Police and major 
          supermarkets could reduce the 
          2,871 incidents recorded there.
      </p>
    </div>

    <div style="flex:1; min-width:220px; background:white;
                border-top: 4px solid #3182ce;
                padding:15px; border-radius:8px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.06);">
      <p style="font-size:24px; margin:0;">4️⃣</p>
      <p style="font-size:14px; font-weight:bold; color:#1a1a2e; margin:8px 0 4px 0;">
          Use This Model Operationally
      </p>
      <p style="font-size:12px; color:#666; line-height:1.7; margin:0;">
          The Random Forest model predicts 
          crime type from location and time 
          with 20.46% improvement over baseline. 
          Integrating this into daily briefings 
          could help officers anticipate — 
          not just respond to — crime.
      </p>
    </div>

  </div>

  <div style="background: linear-gradient(135deg, #1a1a2e, #16213e);
              padding: 25px 30px; border-radius: 12px; color: white;
              text-align: center;">
    <p style="font-size: 16px; font-weight: bold; color: #68d391; margin: 0 0 10px 0;">
        🎓 Final Conclusion
    </p>
    <p style="font-size: 14px; color: white; line-height: 1.8; margin: 0 0 15px 0;">
        Crime in Norfolk is <b>not random</b>. It is spatial, seasonal 
        and predictable. With 60.48% of crimes going unresolved, 
        the case for data-driven policing has never been stronger. 
        This analysis shows that open data, Python and machine learning 
        can provide Norfolk Constabulary with actionable intelligence — 
        at zero cost.
    </p>
    <div style="border-top: 1px solid rgba(255,255,255,0.1); 
                padding-top: 12px; margin-top: 5px;">
      <p style="font-size: 12px; color: #a0aec0; margin: 0;">
          NBS-7143B Applied AI & Business Analytics &nbsp;|&nbsp; 
          University of East Anglia Norwich &nbsp;|&nbsp;
          Data: data.police.uk &nbsp;|&nbsp; 
          Mar 2025 – Mar 2026 &nbsp;|&nbsp; 
          69,103 records
      </p>
    </div>
  </div>

</div>
"""))

HTML(value='\n<div style="font-family: Arial; margin: 20px 0;">\n\n  <h2 style="font-size: 22px; color: #1a1a2…